In [1]:
import sys
from pyprojroot import here
sys.path.append(str(here()))

In [2]:
import math
import numpy as np
from src.utils.db_connect import get_db_connection

In [ ]:
N_STEPS = 3000
N_SIMS = 40

In [4]:
con = get_db_connection(read_only=True)

run_ids_query = f"""
SELECT run_id 
FROM SIMULATION_STEPS
GROUP BY run_id
HAVING COUNT(step_idx) = {N_STEPS}
ORDER BY random()
LIMIT {N_SIMS};
"""

single_run_query = """
SELECT 
    step_idx,
    bead_positions,
    state,
    target_theta
FROM SIMULATION_STEPS
WHERE run_id = ?
ORDER BY step_idx;
"""

In [5]:
selected_run_ids = con.execute(run_ids_query).df()["run_id"].tolist()
print(f"Successfully selected {len(selected_run_ids)} random run IDs to analyze!\n")

Successfully selected 300 random run IDs to analyze!



<hr>

#### **SNR Definition**

The results of this SNR are based off of this paper: [AutoStepFinder](https://www.cell.com/patterns/fulltext/S2666-3899%2821%2900082-9?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS2666389921000829%3Fshowall%3Dtrue)

##### SNR is defined as the following

$$
SNR = \frac{0.5 \times \Delta}{\sigma}
$$

Where

$$
\begin{aligned}
\Delta &= \text{Step Size} \\
\sigma &= \text{Noise Intensity (Standard Deviation of background noise)}
\end{aligned}
$$

<hr>

##### **Estimating the Noise Intensity ($\sigma$)**

The measured signal (bead positions) is modeled as a combination of an underlying step signal (target state trajectory) and additive noise:

$$
X_i = S_i + N_i
$$

where $S_i$ represents the underlying state and $N_i$ is independent, zero-mean measurement noise with constant variance measurement noise.

However, $N_i$ is not directly observable. To isolate the noise contribution, we consider first differences:

$$
\begin{aligned}
\Delta X &= X_{i+1} - X_i \\
         &= (S_{i+1} - S_i) + (N_{i+1} - N_i)
\end{aligned}
$$

This operation is equivalent to a discrete derivative, which acts as a high-pass filter: it suppresses slow variations in the signal (e.g., state persistence or smooth drift) and emphasizes fast fluctuations.

Away from transition points, the signal is approximately constant over one time step, so:

$$
\Delta X \approx N_{i+1} - N_i.
$$

Thus, the differenced signal is dominated by noise increments. Assuming independent, zero-mean measurement noise with constant variance noise:

$$
\mathrm{Var}(\Delta X)
= \mathrm{Var}(N_{i+1}) + \mathrm{Var}(N_i)
= 2\sigma^2.
$$

Taking square roots gives:

$$
\mathrm{std}(\Delta X) = \sqrt{2}\,\sigma.
$$

Therefore, the noise intensity can be estimated as:

$$
\sigma = \frac{\mathrm{std}(\Delta X)}{\sqrt{2}}
$$

<hr>

In [6]:
def SNR(delta, SD):
    return (0.5 * delta) / SD


def NOISE_SD(trajectory):
    diffs = np.diff(trajectory)
    return np.std(diffs) / math.sqrt(2)


def CHANGE_POINTS(targets):
    targets = np.asarray(targets)
    change_mask = targets[1:] != targets[:-1]
    return np.where(change_mask)[0] + 1


def STEP_SIZES(targets, change_indices):
    targets = np.asarray(targets)
    if len(change_indices) == 0:
        return np.array([])
    levels_before = targets[change_indices - 1]
    levels_after = targets[change_indices]
    return np.abs(levels_after - levels_before)

In [ ]:
results = {}  # [id][state] ==> SNR ==> ["avg"][state] ==> avg snr
for run_idx in selected_run_ids: 
    df = con.execute(single_run_query, [run_idx]).df()

    SD_noise = NOISE_SD(df["bead_positions"])
    cp_indices = CHANGE_POINTS(df["target_theta"])
    deltas = STEP_SIZES(df["target_theta"], cp_indices)

    snr_collector = {0: [], 1: [], 2: [], 3: []}
    for i, cp_index in enumerate(cp_indices):
        previous_index = max(0, cp_index - 1)
        cp_state = int(df["state"].iloc[previous_index])
        step_snr = SNR(deltas[i], SD_noise)
        snr_collector[cp_state].append(step_snr)

    results[run_idx] = {}  # Store average SNR per state
    for state, snr_list in snr_collector.items():
        if len(snr_list) > 0:
            results[run_idx][state] = np.mean(snr_list)
        else:
            results[run_idx][state] = None

<hr>

### Analysis of 'None' States in SNR Calculations

**Conclusion: The presence of 'None' values is expected, mathematically correct, and does not negatively impact our SNR analysis.**

#### 1. The Core Logic of Transitions
By definition, a step size requires a transition. If a change point never occurs and the state never changes, it is logically impossible to calculate a delta. Because AutoStepfinder identifies transitions by comparing a plateau to both its left and its right, it cannot place a change point at the very end of a trace where no future data exists. Consequently, the final state of a trajectory yields no delta, resulting in an uncalculable SNR ('None').

#### 2. The Influence of Trajectory Length
The occurrence of these 'None' values is highly dependent on how long the recording runs:
* **Short Trajectories (~2,000 steps):** In these shorter windows, we clearly observe 'None' values because the recording cuts off while the motor is actively lingering in its final state.
* **Long Trajectories (~10,000 steps):** When the step count reaches this scale, we do not see any 'None' values. In other words, statistically at this step count they are exceptionally rare. Over these extended timelines, the motor has sufficient time to complete its cycles, meaning every single state eventually experiences a transition before the long recording finishes.

#### 3. Why the State 3 Distribution Makes Biological Sense
In our ~2,000 step trajectories, 25% of all runs (10 out of 40) end permanently in State 3, while 0% end in States 0, 1, or 2. This asymmetry is highly meaningful:
* **Validates Code Logic:** If 'None' values were scattered randomly across all states, it would point to a glitch in our indexing or state-tracking logic. 
* **Targeted Stalling:** The fact that only State 3 captures these trailing ends indicates that the F1-ATPase motor preferentially stalls or enters an inactive dwell at this state. Based off our stochastic matrix, this seems to be a reasonable conclusion.

#### 4. Impact on Global SNR Averages
Because our aggregation pipeline utilizes a 'None' guard, these final destination states are cleanly omitted from the calculations. The remaining 30 runs where the motor successfully stepped out of State 3 provide more than enough statistical power to calculate a robust, accurate average SNR for State 3 transitions. No data is corrupted, and no manual filtering is required.

In [8]:
print("--- 'None' Counts (Final Destination States) Per State ---")

for state in [0, 1, 2, 3]:
    
    none_count = sum(
        1 for run_id in results 
        if run_id != "average" and results[run_id][state] is None
    )
    
    total_runs = len([run_id for run_id in results if run_id != "average"])
    
    print(f"State {state}: {none_count} Nones out of {total_runs} total runs ({none_count/total_runs*100:.1f}%)")

print("-" * 50)

--- 'None' Counts (Final Destination States) Per State ---
State 0: 0 Nones out of 300 total runs (0.0%)
State 1: 0 Nones out of 300 total runs (0.0%)
State 2: 1 Nones out of 300 total runs (0.3%)
State 3: 22 Nones out of 300 total runs (7.3%)
--------------------------------------------------


In [9]:
results["average"] = {}
for state in [0, 1, 2, 3]:
    state_snrs = [
        results[run_id][state]
        for run_id in results
        if run_id != "average" and results[run_id][state] is not None
    ]

    if state_snrs:
        results["average"][state] = np.mean(state_snrs)
    else:
        results["average"][state] = None

In [10]:
print("--- Global State Averages ---")
print(results["average"])

--- Global State Averages ---
{0: np.float64(2.718263074986457), 1: np.float64(2.965216100206015), 2: np.float64(3.566575729726203), 3: np.float64(1.272441357758919)}
